In [1]:
from datasets import load_dataset
import pandas as pd

dataset = load_dataset("ml-hub/flipkart-reviews-dataset")

df = pd.concat(
    [split.to_pandas() for split in dataset.values()],
    ignore_index=True
)

print("Dataset shape:", df.shape)
print("Columns:", df.columns.tolist())

df.head()

c:\Users\KAYSAN\anaconda3\envs\ml_dl_env\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Dataset shape: (40000, 7)
Columns: ['product_name', 'product_price', 'Rate', 'Review', 'text', 'labels', 'sentiment_code']


,product_name,product_price,Rate,Review,text,labels,sentiment_code
0,Candes 12 L Room/Personal Air Cooler??????(Whi...,3999.0,5.0,super!,great cooler excellent air flow and for this p...,positive,2
1,Candes 12 L Room/Personal Air Cooler??????(Whi...,3999.0,5.0,awesome,best budget 2 fit cooler nice cooling,positive,2
2,Candes 12 L Room/Personal Air Cooler??????(Whi...,3999.0,3.0,fair,the quality is good but the power of air is de...,positive,2
3,Candes 12 L Room/Personal Air Cooler??????(Whi...,3999.0,1.0,useless product,very bad product its a only a fan,negative,0
4,Candes 12 L Room/Personal Air Cooler??????(Whi...,3999.0,3.0,fair,ok ok product,neutral,1


In [2]:
print("Missing values:\n", df.isnull().sum())
print("Duplicate rows:", df.duplicated().sum())

print("\nSample reviews:")
print(df[["product_name", "Rate", "Review", "text"]].head())

Missing values:
 product_name      0
product_price     1
Rate              1
Review            0
text              0
labels            0
sentiment_code    0
dtype: int64
Duplicate rows: 4147

Sample reviews:
                                        product_name  Rate           Review  \
0  Candes 12 L Room/Personal Air Cooler??????(Whi...   5.0           super!   
1  Candes 12 L Room/Personal Air Cooler??????(Whi...   5.0          awesome   
2  Candes 12 L Room/Personal Air Cooler??????(Whi...   3.0             fair   
3  Candes 12 L Room/Personal Air Cooler??????(Whi...   1.0  useless product   
4  Candes 12 L Room/Personal Air Cooler??????(Whi...   3.0             fair   

                                                text  
0  great cooler excellent air flow and for this p...  
1              best budget 2 fit cooler nice cooling  
2  the quality is good but the power of air is de...  
3                  very bad product its a only a fan  
4                                      ok 

In [3]:
# Check exact duplicate rows
print("Exact duplicate rows:", df.duplicated().sum())

# Check duplicate review texts
print("Duplicate review texts:", df["text"].duplicated().sum())

# Check duplicate product-review combinations
print(
    "Duplicate product-review pairs:",
    df.duplicated(subset=["product_name", "text"]).sum()
)

# Inspect repeated examples
df[df.duplicated(keep=False)].head(10)

Exact duplicate rows: 4147
Duplicate review texts: 18320
Duplicate product-review pairs: 11506


,product_name,product_price,Rate,Review,text,labels,sentiment_code
92,MAHARAJA WHITELINE 65 L Desert Air Cooler?????...,7999.0,5.0,highly recommended,awesome,positive,2
93,MAHARAJA WHITELINE 65 L Desert Air Cooler?????...,7999.0,5.0,fabulous!,good product,positive,2
95,MAHARAJA WHITELINE 65 L Desert Air Cooler?????...,7999.0,5.0,great product,very good,positive,2
100,MAHARAJA WHITELINE 65 L Desert Air Cooler?????...,7999.0,5.0,mind-blowing purchase,awesome product,positive,2
113,MAHARAJA WHITELINE 65 L Desert Air Cooler?????...,7999.0,4.0,good quality product,good,positive,2
125,MAHARAJA WHITELINE 65 L Desert Air Cooler?????...,7999.0,5.0,perfect product!,good product,positive,2
136,MAHARAJA WHITELINE 65 L Desert Air Cooler?????...,7999.0,4.0,really nice,good product,positive,2
137,MAHARAJA WHITELINE 65 L Desert Air Cooler?????...,7999.0,5.0,wonderful,good,positive,2
141,MAHARAJA WHITELINE 65 L Desert Air Cooler?????...,7999.0,5.0,super!,good product,positive,2
150,MAHARAJA WHITELINE 65 L Desert Air Cooler?????...,7999.0,4.0,really nice,good product,positive,2


In [4]:
# Remove exact duplicate rows only
df_clean = df.drop_duplicates().reset_index(drop=True)

print("Original dataset:", df.shape)
print("Cleaned dataset:", df_clean.shape)
print("Remaining exact duplicates:", df_clean.duplicated().sum())

Original dataset: (40000, 7)
Cleaned dataset: (35853, 7)
Remaining exact duplicates: 0


In [5]:
print("Unique products:", df_clean["product_name"].nunique())
print("Unique review texts:", df_clean["text"].nunique())

print("\nRating distribution:")
print(df_clean["Rate"].value_counts(dropna=False).sort_index())

print("\nMost common reviews:")
print(df_clean["text"].value_counts().head(10))

Unique products: 205
Unique review texts: 21680

Rating distribution:
Rate
1.0     4377
2.0     1168
3.0     2746
4.0     6954
5.0    20607
NaN        1
Name: count, dtype: int64

Most common reviews:
text
good                 1597
nice                  999
good product          920
nice product          711
super                 456
very good             373
very nice             351
very good product     294
excellent             278
good quality          258
Name: count, dtype: int64


In [6]:
from langchain_core.documents import Document
import pandas as pd

review_docs = []

for _, row in df_clean.iterrows():

    review_text = str(row["text"]).strip()

    if not review_text or review_text.lower() == "nan":
        continue

    product_name = row["product_name"]
    rating = row["Rate"]

    content = (
        f"Product: {product_name}\n"
        f"Rating: {rating if pd.notna(rating) else 'Not available'}\n"
        f"Review: {review_text}"
    )

    review_docs.append(
        Document(
            page_content=content,
            metadata={
                "product_name": str(product_name),
                "rating": (
                    float(rating)
                    if pd.notna(rating)
                    else None
                ),
                "sentiment": str(row["labels"])
            }
        )
    )

print("Total documents created:", len(review_docs))

Total documents created: 35853


In [7]:
print(review_docs[0].page_content)
print(review_docs[0].metadata)

Product: Candes 12 L Room/Personal Air Cooler??????(White, Black, Elegant High Speed-Honey Comb Cooling Pad & Ice Chamber, Blower)
Rating: 5.0
Review: great cooler excellent air flow and for this price its so amazing and unbelievablejust love it
{'product_name': 'Candes 12 L Room/Personal Air Cooler??????(White, Black, Elegant High Speed-Honey Comb Cooling Pad & Ice Chamber, Blower)', 'rating': 5.0, 'sentiment': 'positive'}


In [9]:
import json

output_file = "flipkart_review_documents.json"

documents_data = [
    {
        "page_content": doc.page_content,
        "metadata": doc.metadata
    }
    for doc in review_docs
]

with open(output_file, "w", encoding="utf-8") as file:
    json.dump(
        documents_data,
        file,
        ensure_ascii=False,
        indent=2
    )

print(f"Successfully saved {len(documents_data)} documents!")
print("File:", output_file)

Successfully saved 35853 documents!
File: flipkart_review_documents.json
